[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/02.%20Parte%202/15.%20Clase%2015/13Class%20NB.ipynb)

In [ ]:
!pip install -q yfinance pandas numpy matplotlib seaborn scipy scikit-learn statsmodels cvxpy pyomo

# Clase 15: Optimización convexa (CVXPY) y programación estocástica (Pyomo)

[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres), 

*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)

+ Departamento de Matemáticas y Física
+ dsanchez@iteso.mx
+ Tel. 3669-34-34 Ext. 3069
+ Oficina: Cubículo 4, Edificio J, 2do piso

# 1. Motivación

En esta clase combinamos dos herramientas de optimización:

1. **CVXPY** — para problemas de optimización convexa (portafolios) bajo el enfoque de **Programación Convexa Disciplinada** (DCP).
2. **Pyomo** — para **programación matemática general**, incluyendo problemas lineales enteros mixtos (MILP) como el problema de dieta.

### ¿Por qué dos herramientas?

| Característica | CVXPY | Pyomo |
|---------------|-------|-------|
| **Enfoque** | Optimización convexa (DCP) | Programación matemática general |
| **Variables** | Continuas | Continuas, enteras, binarias |
| **Verificación** | Verifica convexidad automáticamente | No verifica convexidad |
| **Solvers** | ECOS, SCS, MOSEK | GLPK, CBC, Gurobi, CPLEX |
| **Uso ideal** | Portafolios, regresión, SVM | Planificación, scheduling, MILP |

> **Regla práctica**: si el problema es convexo con variables continuas, usa **CVXPY**. Si necesitas variables enteras o el problema no es convexo, usa **Pyomo**.

In [ ]:
#importar los paquetes que se van a usar
import pandas as pd
import numpy as np
import datetime
from datetime import datetime
import scipy.stats as stats
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.covariance as skcov
import cvxpy as cp
%matplotlib inline
pd.set_option('display.notebook_repr_html', True)
pd.set_option('display.max_columns', 6)
pd.set_option('display.max_rows', 10)
pd.set_option('display.width', 78)
pd.set_option('precision', 3)
#Funciones para portafolios
import portfolio_func
from pyomo.environ import *
infinity = float('inf')
import statsmodels.api as sm

In [2]:
assets = ['AAPL','MSFT','AA','AMZN','KO','QAI']
closes = portfolio_func.get_historical_closes(assets, '2025-01-01', '2025-03-27')

In [3]:
daily_returns=portfolio_func.calc_daily_returns(closes)
huber = sm.robust.scale.Huber()
#Mean and standar deviation returns
returns_av, scale = huber(daily_returns)

In [ ]:
# Portafolio con restricción de riesgo — CVXPY (DCP)
#
# maximizar   μ' w              (rendimiento esperado)
# sujeto a    w' Q w ≤ max_risk (restricción cuadrática de riesgo)
#             Σ wᵢ = 1          (presupuesto)
#             wᵢ ≥ 0            (sin ventas en corto)
#
# Verificación DCP:
#   - Objetivo: μ'w es afín → cóncavo ✓ (maximización)
#   - cp.quad_form(w, Q) es convexa ≤ constante ✓
#   - cp.sum(w) == 1 es afín ✓
#   - w >= 0 es afín ✓

n_assets = len(daily_returns.columns)
mu_port = returns_av
Q_port = skcov.ShrunkCovariance().fit(daily_returns).covariance_
max_risk = 0.0001  # Tolerancia máxima de riesgo (varianza)

w_port = cp.Variable(n_assets)
objective = cp.Maximize(mu_port @ w_port)
constraints = [
    cp.quad_form(w_port, Q_port) <= max_risk,
    cp.sum(w_port) == 1,
    w_port >= 0
]
prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.ECOS)

print(f"Status: {prob.status}")
print(f"Rendimiento esperado diario: {prob.value:.6f}")
print(f"\nAsignación óptima:")
for name, val in zip(daily_returns.columns, w_port.value):
    print(f"  {name}: {val:.4f}")

---

## Pyomo: Programación matemática para problemas no convexos y enteros

**Pyomo** (Python Optimization Modeling Objects) es un framework algebraico de modelado que permite formular problemas de optimización de forma declarativa. A diferencia de CVXPY, Pyomo **no se limita a problemas convexos**: soporta variables enteras, binarias, restricciones no lineales y modelos abstractos parametrizados.

### El problema de dieta (MILP)

Un ejemplo clásico de programación lineal entera mixta es el **problema de dieta**: encontrar la combinación de alimentos de menor costo que satisfaga requerimientos nutricionales mínimos.

$$
\min_{x} \sum_{i \in F} c_i \, x_i
$$

sujeto a:

$$
N_j^{\min} \leq \sum_{i \in F} a_{ij} \, x_i \leq N_j^{\max}, \quad \forall \, j \in N
$$

$$
\sum_{i \in F} V_i \, x_i \leq V^{\max}
$$

$$
x_i \in \mathbb{Z}_{\geq 0}, \quad \forall \, i \in F
$$

donde $c_i$ es el costo por porción del alimento $i$, $a_{ij}$ es el contenido del nutriente $j$ en el alimento $i$, y $x_i$ es el número (entero) de porciones.

> **¿Por qué Pyomo y no CVXPY?** Las variables $x_i$ son **enteras** ($\mathbb{Z}_{\geq 0}$), lo que hace que el problema sea un MILP. CVXPY puede manejar variables enteras con `cp.Variable(integer=True)`, pero Pyomo es más natural para modelos abstractos con datos externos (`dietdata.dat`) y solvers especializados como GLPK.

### Ejecución del problema de dieta

El modelo está definido en `diet.py` y los datos en `dietdata.dat`. Se resuelve con el solver GLPK:

In [ ]:
!cat dietdata.dat

In [ ]:
!pyomo solve --solver=glpk diet.py dietdata.dat

In [ ]:
!cat results.yml

# 2. Uso de Pandas para descargar datos de precios de cierre

Usamos `yfinance` para descargar precios de cierre ajustados directamente de Yahoo Finance. La función `get_historical_closes()` definida en `portfolio_func.py` encapsula esta descarga.

# 3. Formulación del riesgo de un portafolio y simulación Montecarlo

In [ ]:
r=0.0001
results_frame = portfolio_func.sim_mont_portfolio(daily_returns,100000,r)

In [ ]:
#Sharpe Ratio
max_sharpe_port = results_frame.iloc[results_frame['Sharpe'].idxmax()]
#Menor SD
min_vol_port = results_frame.iloc[results_frame['SD'].idxmin()]

In [ ]:
plt.scatter(results_frame.SD,results_frame.Returns,c=results_frame.Sharpe,cmap='RdYlBu')
plt.xlabel('Volatility')
plt.ylabel('Returns')
plt.colorbar()
#Sharpe Ratio
plt.scatter(max_sharpe_port[1],max_sharpe_port[0],marker=(5,1,0),color='r',s=1000);
#Menor SD
plt.scatter(min_vol_port[1],min_vol_port[0],marker=(5,1,0),color='g',s=1000);

In [ ]:
pd.DataFrame(max_sharpe_port)

In [ ]:
pd.DataFrame(min_vol_port)

# 4. Optimización de portafolios

In [ ]:
N=5000
results_frame_optim = portfolio_func.optimal_portfolio(daily_returns,N,r)

In [ ]:
#Montecarlo
plt.scatter(results_frame.SD,results_frame.Returns,c=results_frame.Sharpe,cmap='RdYlBu')
plt.xlabel('Volatility')
plt.ylabel('Returns')
plt.colorbar()
#Markowitz
plt.plot(results_frame_optim.SD, results_frame_optim.Returns, 'b-o');

In [ ]:
#Sharpe Ratio
max_sharpe_port_optim = results_frame_optim.iloc[results_frame_optim['Sharpe'].idxmax()]
#Menor SD
min_vol_port_optim = results_frame_optim.iloc[results_frame_optim['SD'].idxmin()]

In [ ]:
#Markowitz
plt.scatter(results_frame_optim.SD,results_frame_optim.Returns,c=results_frame_optim.Sharpe,cmap='RdYlBu');
plt.xlabel('Volatility')
plt.ylabel('Returns')
plt.colorbar()
#Sharpe Ratio
plt.scatter(max_sharpe_port_optim[1],max_sharpe_port_optim[0],marker=(5,1,0),color='r',s=1000);
#SD
plt.scatter(min_vol_port_optim[1],min_vol_port_optim[0],marker=(5,1,0),color='g',s=1000);

In [ ]:
pd.DataFrame(max_sharpe_port_optim)

In [ ]:
pd.DataFrame(min_vol_port_optim)